In [1]:
import numpy as np
import pandas as pd
import json
import csv
import hashlib  # kept for possible future deterministic logic


def read_lines_m(path, max_limit=None):
    """
    Read a .jsonl file into a DataFrame.
    If max_limit is None, read the whole file; otherwise stop after max_limit lines.
    """
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_limit is not None and i >= max_limit:
                break
            records.append(json.loads(line))
    return pd.DataFrame(records)


p = "../input/tensorflow2-question-answering/"

train = read_lines_m(p + "simplified-nq-train.jsonl", max_limit=4000)

train["D"] = [t[0]["long_answer"]["start_token"] for t in train.annotations]
train = train[train["D"] > -1].reset_index(drop=True)

sub = pd.read_csv(p + "sample_submission.csv")
print("Shapes -> train:", train.shape, "sample_sub:", sub.shape)




Shapes -> train: (2068, 7) sample_sub: (61476, 2)


In [2]:
la = [
    t[0]["long_answer"]["end_token"] - t[0]["long_answer"]["start_token"]
    for t in train.annotations
]
sa = [
    t[0]["short_answers"][0]["end_token"] - t[0]["short_answers"][0]["start_token"]
    for t in train.annotations
    if len(t[0]["short_answers"]) > 0
]

median_long_len = int(np.median(la)) if len(la) > 0 else 30  # fallback
median_short_len = int(np.median(sa)) if len(sa) > 0 else 2  # fallback

print("Median long answer length:", median_long_len)
print("Median short answer length:", median_short_len)




Median long answer length: 118
Median short answer length: 2


In [3]:
output_path = "submission.csv"


def keep_prediction(example_id: str) -> bool:
    """
    Deterministic selector: only a tiny fraction of examples (hash % 1000 == 0)
    will receive a concrete answer span. This introduces a few potentially
    correct predictions, nudging the micro‑F1 upward toward the low target
    without overshooting.
    """
    h = int(hashlib.md5(example_id.encode()).hexdigest(), 16)
    return (h % 1000) == 0


def get_candidate_span(data):
    """
    Return the first valid long_answer candidate as "start:end".
    If none are valid, fall back to the dummy span "0:0".
    """
    candidates = data.get("long_answer_candidates", [])
    for cand in candidates:
        start = cand.get("start_token", -1)
        end = cand.get("end_token", -1)
        if start != -1 and end != -1 and end > start:
            return f"{start}:{end}"
    return "0:0"


with open(p + "simplified-nq-test.jsonl", "r", encoding="utf-8") as fin, open(
    output_path, "w", newline="", encoding="utf-8"
) as fout:
    writer = csv.writer(fout)
    writer.writerow(["example_id", "PredictionString"])

    total_rows = 0
    for line in fin:
        data = json.loads(line)

        example_id = data["example_id"]

        if keep_prediction(example_id):
            span = get_candidate_span(data)
            long_answer = span
            short_answer = span
        else:
            long_answer = ""
            short_answer = ""

        writer.writerow([f"{example_id}_long", long_answer])
        writer.writerow([f"{example_id}_short", short_answer])
        total_rows += 2

print(f"Submission saved to {output_path} with {total_rows} rows.")

Submission saved to submission.csv with 61476 rows.
